In [1]:
import os
from pathlib import Path

if Path.cwd().name == "examples":
    os.chdir(Path.cwd().parent)

from hybrid_rag import providers
from hybrid_rag.config import settings

In [2]:
import asyncio
import functools
import logging
import math
from datetime import datetime, timezone
from email.utils import parsedate_to_datetime

import httpx
import openai
from tenacity import before_sleep_log, retry, retry_if_exception, stop_after_attempt

logger = logging.getLogger("host.models")


def is_transient(exc: BaseException) -> bool:
    """True kalau mengirim ulang permintaan yang sama bisa berhasil."""
    if isinstance(exc, httpx.HTTPStatusError):  # jalur rerank
        return exc.response.status_code in (408, 429) or exc.response.status_code >= 500
    if isinstance(exc, (httpx.TransportError, TimeoutError)):  # TimeoutError: stream macet di langchain-openai
        return True
    if isinstance(exc, (openai.APIConnectionError, openai.APITimeoutError)):
        return True
    return isinstance(exc, openai.APIStatusError) and (exc.status_code in (408, 409, 429) or exc.status_code >= 500)

def seconds(value) -> float | None:
    try:
        delay = float(value)
    except (TypeError, ValueError):
        try:
            delay = (parsedate_to_datetime(str(value)).astimezone(timezone.utc) - datetime.now(timezone.utc)).total_seconds()
        except (TypeError, ValueError, IndexError, OverflowError):
            return None
    return max(0.0, float(math.ceil(delay)))


def wait_seconds(state) -> float:
    """`retry_after` dari provider (body JSON, lalu header) kalau ada; kalau tidak 1s, 2s, 4s, maksimal 8s."""
    exc = state.outcome.exception()
    body = getattr(exc, "body", None)
    headers = getattr(getattr(exc, "response", None), "headers", None)
    delay = seconds(body.get("retry_after")) if isinstance(body, dict) else None
    if delay is None and headers is not None:
        delay = seconds(headers.get("retry-after"))
    return delay if delay is not None else min(8.0, 2.0 ** (state.attempt_number - 1))

def with_retry(attempts):
    def decorate(func):
        def policy(*args):
            return retry(
                stop=stop_after_attempt(max(1, attempts(*args))),
                wait=wait_seconds,
                retry=retry_if_exception(is_transient),
                before_sleep=before_sleep_log(logger, logging.WARNING),
                reraise=True,
            )(func)

        if asyncio.iscoroutinefunction(func):
            @functools.wraps(func)
            async def awrapper(*args, **kwargs):
                return await policy(*args)(*args, **kwargs)
            return awrapper

        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            return policy(*args)(*args, **kwargs)
        return wrapper

    return decorate

per_instance = with_retry(lambda self, *_: self.retry_attempts)

In [3]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

class RetryingChat(ChatOpenAI):
    retry_attempts: int = 1
    @per_instance
    def _generate_with_cache(self, *args, **kwargs):
        return super()._generate_with_cache(*args, **kwargs)
    @per_instance
    async def _agenerate_with_cache(self, *args, **kwargs):
        return await super()._agenerate_with_cache(*args, **kwargs)

class RetryingEmbeddings(OpenAIEmbeddings):
    retry_attempts: int = 1
    @per_instance
    def embed_documents(self, *args, **kwargs):
        return super().embed_documents(*args, **kwargs)
    @per_instance
    async def aembed_documents(self, *args, **kwargs):
        return await super().aembed_documents(*args, **kwargs)
    @per_instance
    def embed_query(self, *args, **kwargs):
        return super().embed_query(*args, **kwargs)
    @per_instance
    async def aembed_query(self, *args, **kwargs):
        return await super().aembed_query(*args, **kwargs)


In [4]:

def get_llm(model_id, *, base_url, api_key, timeout, max_retries, max_tokens, **model_kwargs):
    return RetryingChat(
        model=model_id, base_url=base_url, api_key=api_key, timeout=timeout, max_tokens=max_tokens,
        max_retries=0, retry_attempts=max_retries, **model_kwargs,
    )

def get_vlm(model_id, *, base_url, api_key, timeout, max_retries, max_tokens=None):
    return get_llm(model_id, base_url=base_url, api_key=api_key, timeout=timeout, max_retries=max_retries, max_tokens=max_tokens)

def get_embeddings(model_id, *, base_url, api_key, timeout, max_retries):
    return RetryingEmbeddings(
        model=model_id, base_url=base_url, api_key=api_key, timeout=timeout,
        max_retries=0, retry_attempts=max_retries,
        model_kwargs={"encoding_format": "float"},
        check_embedding_ctx_length=False,
    )

def get_reranker(model_id, *, base_url, api_key, timeout, max_retries):
    @with_retry(lambda *_: max_retries)
    async def post(payload):
        async with httpx.AsyncClient(timeout=timeout) as client:
            response = await client.post(base_url, json=payload, headers={"Authorization": f"Bearer {api_key}"})
            response.raise_for_status()
            return response.json()

    async def rerank(query, documents, top_n=None):
        if top_n is not None:
            top_n = min(top_n, len(documents))
        data = await post({"model": model_id, "query": query, "documents": documents, "top_n": top_n})
        return data["results"]

    return rerank


# Daftarkan ke hybrid_rag

In [5]:
providers.configure(get_llm=get_llm, get_vlm=get_vlm, get_embeddings=get_embeddings, get_reranker=get_reranker)

# LLM

In [6]:
settings.require("extract")
llm = providers.get_llm(
    settings.extract_llm_model,
    base_url=settings.extract_llm_binding_host,
    api_key=settings.extract_llm_binding_api_key,
    timeout=settings.extract_llm_timeout,
    temperature=0.0,
)

# VLM

In [7]:
import base64
from langchain_core.messages import HumanMessage

settings.require("vlm")
vlm = providers.get_vlm(
    settings.vlm_model,
    base_url=settings.vlm_binding_host,
    api_key=settings.vlm_binding_api_key,
    timeout=settings.vlm_timeout,
)
# image = base64.b64encode(Path("examples/contoh.png").read_bytes()).decode()  # ganti dengan gambar Anda
# message = HumanMessage(content=[
#     {"type": "text", "text": "Jelaskan gambar ini."},
#     {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image}"}},
# ])
# (await vlm.ainvoke([message])).content

# EMBEDDINGS

In [8]:
settings.require("embedding")
embeddings = providers.get_embeddings(
    settings.embedding_model,
    base_url=settings.embedding_binding_host,
    api_key=settings.embedding_binding_api_key,
    timeout=settings.embedding_timeout,
)
# vectors = await embeddings.aembed_documents(["pasal 1", "pasal 2"])
# len(vectors), len(vectors[0])  # (2, EMBEDDING_DIM)

# RERANK

In [9]:
rerank = providers.get_reranker(
    settings.rerank_model,
    base_url=settings.rerank_binding_host,
    api_key=settings.rerank_binding_api_key,
    timeout=settings.rerank_timeout,
)
# await rerank(query="ibu kota Indonesia", documents=["Jakarta adalah ibu kota.", "Pisang itu buah."], top_n=2)

# Ingestion

Skema database dibuat sekali lewat `make db-init`. Lalu parse file dan tulis ke satu workspace.

In [10]:
from hybrid_rag.ingestion import parse_file, workspace_pool as write_pool
from pathlib import Path
path = Path("examples/test1.xlsx")  
async with parse_file(path) as content_list:
    async with write_pool.lease("workspace_c") as rag:
        await rag.insert_content_list(content_list, file_path=path, doc_id="doc-1")

Retrying __main__.RetryingEmbeddings.aembed_documents in 1 seconds as it raised APIConnectionError: Connection error..


# Retrieval

In [10]:
from hybrid_rag.retrieval import QueryParam, retrieve_context

result = await retrieve_context(
    "rekening BNI",
    workspace="workspace_c",
    param=QueryParam(include_citations=True),
)
print(result.context)


Knowledge Graph (Entity) :

```json
{"entity": "Bni", "type": "organization", "description": "Bank Nasional Indonesia, institusi perbankan yang digunakan untuk melakukan transaksi transfer uang."}
{"entity": "Bi-Fast", "type": "method", "description": "Layanan sistem pembayaran yang digunakan dalam transaksi transfer antar bank."}
{"entity": "Marchel Andrian Shevchen", "type": "role", "description": "Penerima transfer dalam transaksi keuangan yang dilakukan melalui aplikasi BNI."}
{"entity": "I Gede Satyananda Gautama", "type": "role", "description": "Pengirim atau pelaku transaksi transfer melalui aplikasi BNI."}
{"entity": "Kepulauan Bangka Belitung", "type": "organization", "description": "Provinsi dengan 7 KoKab (1 Kota dan 6 Kabupaten) yang memiliki 0 KorKotKab dalam jejaring ICCN.<SEP>Provinsi di Indonesia dengan 7 KoKab (1 Kota dan 6 Kabupaten) yang memiliki 0 KorKotKab dalam jejaring ICCN."}
{"entity": "konfirmasi_transfer_bni (image)", "type": "image", "description": "Layar k